In [1]:
import pandas as pd
import google.colab
google.colab.drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/CS1090B/project/data/'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
df = pd.read_parquet(DATA_ROOT + 'moe_data.parquet')
df.head()

,ticker,yes_price,no_price,count,taker_side,created_time,volume,log_volume,target_price_5m,signed_ret_5m,...,probability_down_5minutes_moirai,probability_no_jump_15minutes_moirai,probability_up_15minutes_moirai,probability_down_15minutes_moirai,probability_no_jump_30minutes_moirai,probability_up_30minutes_moirai,probability_down_30minutes_moirai,probability_no_jump_60minutes_moirai,probability_up_60minutes_moirai,probability_down_60minutes_moirai
0,ACPI-24-B2.5,79,21,992,yes,2025-01-06 16:02:37.618186,78368.0,11.269184,79.0,0.0,...,0.04875,0.921485,0.032028,0.046487,0.926274,0.023716,0.050010,0.929172,0.021895,0.048933
1,ACPI-24-B2.5,79,21,500,no,2025-01-07 00:47:25.410839,10500.0,9.259226,78.0,-1.0,...,0.04238,0.930946,0.029492,0.039561,0.933477,0.026275,0.040248,0.938123,0.024355,0.037523
2,ACPI-24-B2.5,77,23,15,no,2025-01-07 00:47:25.410839,345.0,5.846439,78.0,1.0,...,0.04238,0.930946,0.029492,0.039561,0.933477,0.026275,0.040248,0.938123,0.024355,0.037523
3,ACPI-24-B2.5,77,23,576,no,2025-01-07 00:47:25.410839,13248.0,9.491677,78.0,1.0,...,0.04238,0.930946,0.029492,0.039561,0.933477,0.026275,0.040248,0.938123,0.024355,0.037523
4,ACPI-24-B2.5,78,22,1768,no,2025-01-07 00:47:25.410839,38896.0,10.568672,78.0,0.0,...,0.04238,0.930946,0.029492,0.039561,0.933477,0.026275,0.040248,0.938123,0.024355,0.037523


In [5]:
import numpy as np
import pandas as pd
import gc, os, shutil

HORIZONS = [5, 15, 30, 60]
MODELS = ['lgbm', 'lstm', 'mamba', 'moirai']
MODELS_M = ['ftt', 'ctts']
ALL_MODELS = MODELS + MODELS_M
ROLLING_WINDOWS = [50, 1000]


def get_prob_cols(model, horizon):
    if model in MODELS:
        s = f'{horizon}minutes_{model}'
    else:
        s = f'{horizon}m_{model}'
    return [f'probability_down_{s}',
            f'probability_no_jump_{s}',
            f'probability_up_{s}']

# ── Step 2: build features one horizon at a time ────────────────────────────
def build_meta_features_for_horizon(df, horizon):
    target_col = f'jump3_{horizon}m'
    print(f'\n=== horizon = {horizon}m ===')

    # argmax + correctness + coverage (all valid now since we dropped NaN rows)
    print('  1. argmax + correctness...')
    target_vals = df[target_col].values
    correct_cols = []

    for model in ALL_MODELS:
        cols = get_prob_cols(model, horizon)
        vals = df[cols].values
        # No NaNs anymore — straightforward argmax
        argmax_idx = vals.argmax(axis=1)
        argmax_label = (argmax_idx - 1).astype(np.float32)
        correct = (argmax_label == target_vals).astype(np.float32)
        # Target itself may still be NaN at end-of-ticker; carry that through
        correct[np.isnan(target_vals)] = np.nan

        correct_col = f'__correct_{model}_{horizon}m'
        df[correct_col] = correct
        correct_cols.append((model, correct_col))

        # All rows have predictions now, so has_* is just 1 — skip storing it
        # (or keep as a sanity check; up to you)

    # Rolling accuracy
    print('  2. rolling accuracy (2 windows)...')
    g = df.groupby('ticker', observed=True, sort=False)
    for model, correct_col in correct_cols:
        shifted = g[correct_col].shift(1)
        shifted_grouped = shifted.groupby(df['ticker'],
                                           observed=True, sort=False)
        for w in ROLLING_WINDOWS:
            out = (shifted_grouped
                   .rolling(w, min_periods=max(5, w // 10))
                   .mean()
                   .reset_index(level=0, drop=True)
                   .astype(np.float32))
            df[f'rolling_acc_{model}_w{w}_{horizon}m'] = out

    # Expert agreement
    print('  3. expert agreement...')
    all_proba = np.stack([df[get_prob_cols(m, horizon)].values
                          for m in ALL_MODELS], axis=1).astype(np.float32)
    argmax_per_model = all_proba.argmax(axis=-1)
    counts = np.zeros((len(df), 3), dtype=np.int8)
    for cls in range(3):
        counts[:, cls] = (argmax_per_model == cls).sum(axis=1)
    df[f'agreement_{horizon}m'] = (counts.max(axis=1) /
                                    len(ALL_MODELS)).astype(np.float32)
    del all_proba, argmax_per_model, counts
    gc.collect()

    # Cleanup tmp cols
    tmp_cols = [c for c in df.columns if c.startswith('__')]
    df.drop(columns=tmp_cols, inplace=True)
    gc.collect()

    new_cols = [c for c in df.columns
                if (str(horizon) + 'm' in c and
                    ('rolling_' in c or 'agreement_' in c))]
    print(f'  → added {len(new_cols)} features for {horizon}m')
    print(f'  → df mem now: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')


for h in HORIZONS:
    build_meta_features_for_horizon(df, h)


# ── Step 3: save ────────────────────────────────────────────────────────────
print(f'\nFinal df: {df.shape}, '
      f'{df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

OUT_PATH = os.path.join(DATA_ROOT, 'moe_data.parquet')
LOCAL_TMP = '/content/moe_data.parquet'

print('\nWriting locally...')
df.to_parquet(LOCAL_TMP, engine='pyarrow', compression='zstd', index=False)
print(f'Local size: {os.path.getsize(LOCAL_TMP) / 1e9:.2f} GB')

print('Copying to Drive...')
shutil.copy(LOCAL_TMP, OUT_PATH)
print(f'Saved to {OUT_PATH}')


=== horizon = 5m ===
  1. argmax + correctness...
  2. rolling accuracy (2 windows)...
  3. expert agreement...
  → added 13 features for 5m
  → df mem now: 33.80 GB

=== horizon = 15m ===
  1. argmax + correctness...
  2. rolling accuracy (2 windows)...
  3. expert agreement...
  → added 13 features for 15m
  → df mem now: 36.07 GB

=== horizon = 30m ===
  1. argmax + correctness...
  2. rolling accuracy (2 windows)...
  3. expert agreement...
  → added 13 features for 30m
  → df mem now: 38.33 GB

=== horizon = 60m ===
  1. argmax + correctness...
  2. rolling accuracy (2 windows)...
  3. expert agreement...
  → added 13 features for 60m
  → df mem now: 40.60 GB

Final df: (43550123, 187), 40.60 GB

Writing locally...
Local size: 15.00 GB
Copying to Drive...
Saved to /content/drive/MyDrive/CS1090B/project/data/moe_data.parquet
